In [40]:
import os
from torch.utils.data import Dataset
from PIL import Image
import xmltodict
import torch
from torchvision import transforms
class VOCDataset(Dataset):
    def __init__(self,image_folder,label_folder,transform=None):
        self.image_folder = image_folder
        self.label_folder = label_folder
        self.image_names=os.listdir(self.image_folder)
        self.transform = transform
        self.classes_list = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

    
    def __len__(self):
        return len(self.image_names)
    
    def __getitem__(self,index):
        image_name = self.image_names[index]
        image_path=os.path.join(self.image_folder,image_name)
        image=Image.open(image_path).convert('RGB')
        label_name=image_name.split('.')[0]+'.xml'
        label_path=os.path.join(self.label_folder,label_name)
        with open(label_path) as f:
            label_centnet=f.read()
        label_dict=xmltodict.parse(label_centnet)
        target=[]
        objects=label_dict['annotation']['object']
        for object in objects:
            object_name=object['name']
            object_class_id=self.classes_list.index(object_name)
            object_xmax=float(object['bndbox']['xmax'])
            object_xmin=float(object['bndbox']['xmin'])
            object_ymax=float(object['bndbox']['ymax'])
            object_ymin=float(object['bndbox']['ymin'])
            target.append([object_class_id,object_xmin,object_ymin,object_xmax,object_ymax])
        target=torch.tensor(target)
        if self.transform is not None:
            image=self.transform(image)
        return image,target
    

In [41]:
image_folder = '400_data/VOCdevkit/VOC2007/JPEGImages'
label_folder = '400_data/VOCdevkit/VOC2007/Annotations'
train_dataset=VOCDataset(image_folder,label_folder,transforms.ToTensor())
print(train_dataset[10])

(tensor([[[0.1059, 0.1020, 0.0863,  ..., 0.2941, 0.3098, 0.3137],
         [0.0706, 0.0941, 0.0902,  ..., 0.2824, 0.3529, 0.3451],
         [0.1176, 0.1333, 0.0745,  ..., 0.3098, 0.3020, 0.3176],
         ...,
         [0.1569, 0.1490, 0.1529,  ..., 0.1529, 0.1529, 0.1412],
         [0.1333, 0.1294, 0.1255,  ..., 0.1412, 0.1451, 0.1412],
         [0.1569, 0.1529, 0.1412,  ..., 0.1294, 0.1294, 0.1490]],

        [[0.1255, 0.1216, 0.1059,  ..., 0.2078, 0.2235, 0.2275],
         [0.0902, 0.1137, 0.1098,  ..., 0.1961, 0.2667, 0.2588],
         [0.1333, 0.1490, 0.0902,  ..., 0.2314, 0.2235, 0.2392],
         ...,
         [0.2275, 0.2196, 0.2118,  ..., 0.1608, 0.1608, 0.1490],
         [0.2039, 0.1882, 0.1804,  ..., 0.1490, 0.1529, 0.1608],
         [0.2275, 0.2118, 0.1961,  ..., 0.1373, 0.1490, 0.1725]],

        [[0.2902, 0.2863, 0.2706,  ..., 0.2196, 0.2353, 0.2392],
         [0.2549, 0.2784, 0.2745,  ..., 0.2078, 0.2784, 0.2706],
         [0.3098, 0.3255, 0.2667,  ..., 0.2392, 0.2314, 0